# BMEG 424/524 Tutorial 8: Nearest Neighbours across Embeddings [scRNAseq]

## Biological: Cell similarity
Transcriptome measurements of cells provide gene expression state representations. These states are not random; instead, underlying epigenetic programs and extra-cellular singalling drive controlled expression of transcription factors and gene programs. Accordingly, certain programs define classically known cell types or states like cell cycle, differentiation, and immune functions.

<img src="https://media.springernature.com/full/springer-static/image/art%3A10.1038%2Fs41576-023-00618-5/MediaObjects/41576_2023_618_Fig1_HTML.png" width="35%">

During hematopoiesis - the division of blood-forming stem cells - we observe a continuous gradient of cell states corresponding to different potentials. Naturally, we expect stem cells to share a similar transcriptional state compared to more differentiated cell types like erthrocytes (red blood cells).

![](https://ars.els-cdn.com/content/image/1-s2.0-S0006497120426643-gr1.jpg)

## Technical: How do embeddings represent neighbourhoods?
Raw scRNAseq data lies in ~20000 dimensions (one dimension per measured gene) where gene regulatory networks (biology), noise, and dropout (technical) drive the observed differences. Given that inherent patterns exist in gene regulation we can attempt to represent cells in lower dimensions which stratify cells based on biological states rather than a noisy ~20000 dimension embedding. Glancing over the technical details or normalizing data, and ignoring the possibility of batch effects, PCA, tSNE, and UMAP represent historical and currently visualizations of single cell data. The question we ask here: **do these representations faithfully take the 20000 dimension cells and represent them in 2D?**

While many approaches may exist to answer such a question, we will measure the Jaccard Similarity between neighbourhoods of cells across the different embedding methods. Such an approach ignores information like cell type altogether and simply looks, regardless of the specific cell, if a given cell has the same neighbours one embedding A as in embedding B.

<img src="https://i.sstatic.net/7AHxK.png" width="50%">

Trivially, if an algorithm randomly assigned each 20000 dimensional cell onto a 100 x 100 x 100 3D grid, we would expect a poor performance as neighbours/cells would be suffled randomly.

# Comparing neighbourhood preservation across embeddings

Note: Some code generated with CLAUDE AI.

In [ ]:
!pip install scanpy anndata umap-learn leidenalg -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from itertools import combinations

import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=True)
SEED = 42



## number of nearest neighbors for all comparisons
K = 20

In [ ]:
# Paul et al. 2015 — 2730 murine hematopoietic progenitors, 19 cell types
# Myeloid differentiation: HSC → CMP → GMP/MEP → mature lineages
adata = sc.datasets.paul15()
print(adata)
print(f"\nCell types: {sorted(adata.obs['paul15_clusters'].unique())}")

In [ ]:
# ── PREPROCESSING ─────────────────────────────────────────────────────────
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata.copy()   # store normalized log counts before scaling

# HVG selection (store boolean mask before scaling)
sc.pp.highly_variable_genes(adata, n_top_genes=1000, flavor='seurat')
sc.pl.highly_variable_genes(adata)
hvg_mask = adata.var['highly_variable'].values
print(f"HVGs selected: {hvg_mask.sum()} / {len(hvg_mask)}")

# Scale for PCA
sc.pp.scale(adata, max_value=10)

# PCA (compute enough PCs for all conditions)
sc.tl.pca(adata, n_comps=100, svd_solver='arpack', random_state=SEED)
sc.pl.pca(adata, color=["paul15_clusters"])

# Reference UMAP (n_pcs=30) — used as a consistent 2D layout for all plots
sc.pp.neighbors(adata, n_neighbors=K, n_pcs=30, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
sc.pl.umap(adata, color=["paul15_clusters"])
ref_umap = adata.obsm['X_umap'].copy()
cell_labels = adata.obs['paul15_clusters'].values

print("Preprocessing complete.")

In [ ]:
# ── UTILITY FUNCTIONS ──────────────────────────────────────────────────────

def get_knn_indices(X, k=K):
    """Return (n_cells x k) array of neighbor indices (excludes self)."""
    nn = NearestNeighbors(n_neighbors=k+1, metric='euclidean', n_jobs=-1)
    nn.fit(X)
    indices = nn.kneighbors(X, return_distance=False)[:, 1:]  # drop self
    return indices  # shape: (n_cells, k)


def jaccard_per_cell(idx_a, idx_b):
    """Per-cell Jaccard similarity between two (n_cells x k) neighbor index arrays."""
    n = idx_a.shape[0]
    scores = np.zeros(n)
    for i in range(n):
        a, b = set(idx_a[i]), set(idx_b[i])
        scores[i] = len(a & b) / len(a | b)
    return scores


def mean_jaccard(idx_a, idx_b):
    return jaccard_per_cell(idx_a, idx_b).mean()


def build_jaccard_matrix(nn_dict):
    """Compute pairwise mean Jaccard matrix from dict {label: knn_indices}."""
    labels = list(nn_dict.keys())
    n = len(labels)
    mat = np.eye(n)
    for i, j in combinations(range(n), 2):
        j_val = mean_jaccard(nn_dict[labels[i]], nn_dict[labels[j]])
        mat[i, j] = mat[j, i] = j_val
    return pd.DataFrame(mat, index=labels, columns=labels)


def plot_umap_jaccard(ref_umap, jaccard_scores, title, ax, cmap='plasma'):
    """Scatter cells on reference UMAP, color = per-cell Jaccard vs reference method."""
    sc_plot = ax.scatter(
        ref_umap[:, 0], ref_umap[:, 1],
        c=jaccard_scores, cmap=cmap, s=8, vmin=0, vmax=1, rasterized=True
    )
    ax.set_title(title, fontsize=10, pad=4)
    ax.set_xticks([]); ax.set_yticks([])
    return sc_plot

print("Utilities defined.")

In [ ]:
# ── COMPUTE KNN IN EACH REPRESENTATION ────────────────────────────────────
# All computations use Euclidean distance in the respective feature space.

X_all  = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
X_hvg  = X_all[:, hvg_mask]
X_pca  = adata.obsm['X_pca']   # shape: (n_cells, 50)

print("Computing KNN — all genes...")
nn_all_genes = get_knn_indices(X_all)

print("Computing KNN — HVGs...")
nn_hvg = get_knn_indices(X_hvg)

print("Computing KNN — PCA 10 PCs...")
nn_pca10 = get_knn_indices(X_pca[:, :10])

print("Computing KNN — PCA 30 PCs...")
nn_pca30 = get_knn_indices(X_pca[:, :30])

print("Computing KNN — PCA 50 PCs...")
nn_pca50 = get_knn_indices(X_pca[:, :50])

print("Computing KNN — PCA 100 PCs...")
nn_pca100 = get_knn_indices(X_pca[:, :100])

print("All-gene and PCA neighborhoods done.")

In [ ]:
# ── tSNE: VARYING PERPLEXITY ───────────────────────────────────────────────
# tSNE is will be run on 30 PCs for efficiency.
# Perplexity ~ effective neighborhood size; range typically 5–50.
# We compute tSNE embeddings and then get KNN in the 2D embedded space.

PERPLEXITIES = [5, 30, 100]
tsne_embeddings = {}
nn_tsne = {}

for perp in PERPLEXITIES:
    print(f"  tSNE perplexity={perp}...")
    tsne = TSNE(
        n_components=2,
        perplexity=perp,
        n_iter=1000,
        random_state=SEED,
        init='pca'
    )
    emb = tsne.fit_transform(X_pca[:, :30])
    tsne_embeddings[perp] = emb
    nn_tsne[f'tSNE_perp{perp}'] = get_knn_indices(emb)

print("tSNE done.")

In [ ]:
# ── UMAP: VARYING n_neighbors AND min_dist ─────────────────────────────────
# n_neighbors controls local vs global structure in the fuzzy simplicial complex.
# min_dist controls compactness of the 2D layout.
# KNN is computed in the resulting 2D embedding.

import umap

UMAP_PARAMS = [
    {'n_neighbors': 5,  'min_dist': 0.1,  'label': 'UMAP_n5_d0.1'},
    {'n_neighbors': 15, 'min_dist': 0.1,  'label': 'UMAP_n15_d0.1'},
    {'n_neighbors': 50, 'min_dist': 0.1,  'label': 'UMAP_n50_d0.1'},
    {'n_neighbors': 15, 'min_dist': 0.5,  'label': 'UMAP_n15_d0.5'},
    {'n_neighbors': 15, 'min_dist': 0.01, 'label': 'UMAP_n15_d0.01'},
]

umap_embeddings = {}
nn_umap = {}

for p in UMAP_PARAMS:
    label = p['label']
    print(f"  UMAP {label}...")
    reducer = umap.UMAP(
        n_neighbors=p['n_neighbors'],
        min_dist=p['min_dist'],
        n_components=2,
        random_state=SEED
    )
    emb = reducer.fit_transform(X_pca[:, :30])
    umap_embeddings[label] = emb
    nn_umap[label] = get_knn_indices(emb)

print("UMAP done.")

In [ ]:
# ── ASSEMBLE FULL KNN DICTIONARY ──────────────────────────────────────────
all_nn = {
    'All genes':     nn_all_genes,
    'HVGs':          nn_hvg,
    'PCA 10 PCs':    nn_pca10,
    'PCA 30 PCs':    nn_pca30,
    'PCA 50 PCs':    nn_pca50,
    'PCA 100 PCs':   nn_pca100,
    **{f'tSNE p={p}': nn_tsne[f'tSNE_perp{p}'] for p in PERPLEXITIES},
    **nn_umap
}

print(f"Representations: {list(all_nn.keys())}")

## Data analysis

Lets see how (Jaccard) similar the neighbourhoods are across embeddings.

In [ ]:
print("Building Jaccard matrix (pairwise, all methods)...")
jaccard_df = build_jaccard_matrix(all_nn)
print(jaccard_df.round(3))

In [ ]:
# ── FIGURE 1: Pairwise Jaccard Heatmap ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8,8))

mask = np.zeros_like(jaccard_df.values, dtype=bool)
# no mask — show full symmetric matrix for clarity

sns.heatmap(
    jaccard_df,
    annot=True, fmt='.2f',
    cmap='YlOrRd',
    vmin=0, vmax=1,
    # linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8},
    cbar_kws={'label': f'Mean Jaccard similarity (k={K} neighbors)'}
)

ax.set_title(
    f'Pairwise Mean Jaccard Similarity of k={K} Nearest Neighbor Sets\n'
    'Paul et al. 2015 Hematopoiesis — 2730 cells',
    fontsize=13, pad=12
)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)
plt.tight_layout()
plt.show()
print("\nInterpretation: Values near 1.0 = near-identical neighborhoods; near 0 = distinct topology.")

You can dig deeper into this problem via https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1011288



**What does this mean about existing approaches to visualizing data?** Which is closest to the truth? Are they all bad or is our metric simply faulty?


In [ ]:
# ── FIGURE 2: PCA Scree Plot ──────────────────────────────────────────────
var_exp = adata.uns['pca']['variance_ratio']
cum_var = np.cumsum(var_exp)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(var_exp)+1), var_exp, color='steelblue', alpha=0.8)
axes[0].axvline(10, color='red',    linestyle='--', label='10 PCs')
axes[0].axvline(30, color='orange', linestyle='--', label='30 PCs')
axes[0].axvline(50, color='green',  linestyle='--', label='50 PCs')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].legend()

axes[1].plot(range(1, len(cum_var)+1), cum_var, color='steelblue', linewidth=2)
for n_pc, col, label in zip([10, 30, 50], ['red','orange','green'],
                             ['10 PCs', '30 PCs', '50 PCs']):
    axes[1].axvline(n_pc, color=col, linestyle='--', label=f'{label}: {cum_var[n_pc-1]:.1%}')
axes[1].set_xlabel('Number of PCs')
axes[1].set_ylabel('Cumulative Variance Ratio')
axes[1].set_title('Cumulative Variance Explained')
axes[1].legend()

plt.suptitle('PCA on HVGs (scaled) — Paul15 Hematopoiesis', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── tSNE Embeddings at 3 Perplexity Values ─────────────────────
# Color = cell type label from Paul15

unique_labels = sorted(adata.obs['paul15_clusters'].unique())
palette = sns.color_palette('tab20', n_colors=len(unique_labels))
label_to_color = {l: palette[i] for i, l in enumerate(unique_labels)}
colors = [label_to_color[l] for l in cell_labels]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, perp in zip(axes, PERPLEXITIES):
    emb = tsne_embeddings[perp]
    ax.scatter(emb[:, 0], emb[:, 1], c=colors, s=8, alpha=0.7, rasterized=True)
    ax.set_title(f'tSNE  perplexity={perp}', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

# Legend (shared)
from matplotlib.lines import Line2D
handles = [Line2D([0],[0], marker='o', color='w',
                  markerfacecolor=label_to_color[l], markersize=7, label=l)
           for l in unique_labels]
fig.legend(handles=handles, loc='lower center', ncol=10, fontsize=7,
           bbox_to_anchor=(0.5, -0.12))
plt.suptitle('tSNE Embeddings — Effect of Perplexity on Global vs Local Structure', fontsize=12)
plt.tight_layout()
plt.savefig('tsne_perplexity.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
Perplexity interpretation:
  low  (5)   → tight local clusters, breaks global continuity
  mid  (30)  → standard; balances local/global
  high (100) → smoother, global topology more preserved but clusters blur
""")

In [ ]:
# ── FIGURE 4: UMAP Embeddings — Varying n_neighbors and min_dist ──────────
fig, axes = plt.subplots(1, 5, figsize=(24, 4.5))

for ax, p in zip(axes, UMAP_PARAMS):
    emb = umap_embeddings[p['label']]
    ax.scatter(emb[:, 0], emb[:, 1], c=colors, s=8, alpha=0.7, rasterized=True)
    ax.set_title(f"n_neighbors={p['n_neighbors']}\nmin_dist={p['min_dist']}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

fig.legend(handles=handles, loc='lower center', ncol=10, fontsize=7,
           bbox_to_anchor=(0.5, -0.15))
plt.suptitle(
    'UMAP Embeddings — Effect of n_neighbors (local/global balance) and min_dist (compactness)',
    fontsize=12
)
plt.tight_layout()
plt.savefig('umap_params.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
n_neighbors interpretation:
  small (5)  → very local; many disconnected microclusters
  large (50) → global; broader manifold structure
min_dist interpretation:
  small (0.01) → tightly packed clusters, intra-cluster detail
  large (0.5)  → diffuse layout, inter-cluster relations more visible
""")

In [ ]:
# ── FIGURE 6: Neighbor Overlap Bar Chart ──────────────────────────────────
# Mean Jaccard of each method vs PCA-30, sorted descending.

mean_jac_vs_ref = {
    key: mean_jaccard(ref_nn, all_nn[key])
    for key in compare_keys
}
sorted_keys = sorted(mean_jac_vs_ref, key=mean_jac_vs_ref.get, reverse=True)
sorted_vals = [mean_jac_vs_ref[k] for k in sorted_keys]

bar_colors = ['#2196F3' if 'PCA' in k
              else '#FF5722' if 'tSNE' in k
              else '#4CAF50' if 'UMAP' in k
              else '#9C27B0'
              for k in sorted_keys]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.barh(sorted_keys, sorted_vals, color=bar_colors, alpha=0.85, edgecolor='k', linewidth=0.5)

for bar, val in zip(bars, sorted_vals):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlim(0, 1.0)
ax.set_xlabel(f'Mean Jaccard Similarity vs {ref_key} (k={K})', fontsize=11)
ax.set_title(
    f'Neighborhood Preservation Relative to PCA 30 PCs Reference\n'
    '(Blue=PCA, Orange=tSNE, Green=UMAP, Purple=gene-space)',
    fontsize=12
)
ax.axvline(0.5, linestyle='--', color='gray', alpha=0.6, label='J=0.5')
ax.legend(fontsize=9)
plt.tight_layout()
# plt.savefig('jaccard_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── FIGURE 7: Neighbor Consistency per Cell Type ──────────────────────────
# For each cell type, what is the mean Jaccard vs PCA-30?
# Reveals which cell types are most sensitive to representation choice.

# Compute per-cell Jaccard for all methods vs PCA-30
per_cell_jac = {
    key: jaccard_per_cell(ref_nn, all_nn[key])
    for key in compare_keys
}

cell_type_arr = np.array(cell_labels)

# Build dataframe: rows = cell types, cols = methods
ct_jac = pd.DataFrame(
    {key: pd.Series(vals).groupby(cell_type_arr).mean()
     for key, vals in per_cell_jac.items()}
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    ct_jac,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    vmin=0, vmax=1,
    linewidths=0.4,
    ax=ax,
    annot_kws={'size': 7},
    cbar_kws={'label': f'Mean Jaccard vs PCA 30 PCs (k={K})'}
)
ax.set_title(
    'Per-Cell-Type Neighborhood Preservation vs PCA 30 PCs Reference\n'
    'Green = well-preserved; Red = topology altered',
    fontsize=12, pad=10
)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=9)
plt.tight_layout()
plt.show()
print("""
Biological interpretation:
  Cell types with low Jaccard across methods = poorly resolved by all representations;
  likely transitional/progenitor states with diffuse transcriptional signatures.
  Cell types with high Jaccard = robustly defined; stable to representation choice.
""")

In [ ]:
# ── FIGURE 8: tSNE — Neighbor Change Detail ──────────────────────────────
# How many of the tSNE p=5 vs p=100 neighbors overlap per cell,
# visualized on all three tSNE layouts.

jac_tsne_5_vs_100 = jaccard_per_cell(
    nn_tsne['tSNE_perp5'], nn_tsne['tSNE_perp100']
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, perp in zip(axes, PERPLEXITIES):
    emb = tsne_embeddings[perp]
    sc_plot = ax.scatter(
        emb[:, 0], emb[:, 1],
        c=jac_tsne_5_vs_100, cmap='plasma',
        s=10, vmin=0, vmax=1, rasterized=True
    )
    ax.set_title(f'tSNE perplexity={perp}', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

cbar = fig.colorbar(sc_plot, ax=axes)
cbar.set_label('Per-cell Jaccard: tSNE p=5 vs p=100', fontsize=10)
plt.suptitle(
    'Neighbor Overlap between Extreme Perplexity Values (p=5 vs p=100)\n'
    'Colored on each tSNE layout — shows which cells are most perplexity-sensitive',
    fontsize=11
)
plt.tight_layout()
plt.show()

In [ ]:
# ── FIGURE 9: UMAP — n_neighbors Effect on Neighborhood ──────────────────
# Compare nn_umap for n_neighbors={5,15,50} with min_dist=0.1

jac_umap_5_vs_50 = jaccard_per_cell(
    nn_umap['UMAP_n5_d0.1'], nn_umap['UMAP_n50_d0.1']
)

umap_subset = ['UMAP_n5_d0.1', 'UMAP_n15_d0.1', 'UMAP_n50_d0.1']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, key in zip(axes, umap_subset):
    emb = umap_embeddings[key]
    sc_plot = ax.scatter(
        emb[:, 0], emb[:, 1],
        c=jac_umap_5_vs_50, cmap='plasma',
        s=10, vmin=0, vmax=1, rasterized=True
    )
    ax.set_title(key.replace('_', ' '), fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

cbar = fig.colorbar(sc_plot, ax=axes)
cbar.set_label('Per-cell Jaccard: UMAP n=5 vs n=50 (min_dist=0.1)', fontsize=10)
plt.suptitle(
    'Neighbor Overlap: UMAP n_neighbors=5 vs 50 (min_dist=0.1)\n'
    'Yellow = neighborhood stable; Purple = neighborhood changes with scale',
    fontsize=11
)
plt.tight_layout()
plt.savefig('umap_n_neighbors_comparison.png', dpi=150, bbox_inches='tight')
plt.show()